# TUT: XGBoost centralizado, boosting cíclico y modelos por clúster

Evalúa coordenadas y planta sobre la partición interna 70/15/15 agrupada por posición y conserva los cinco dispositivos más representados.


## 1. Objetivo y protocolo experimental

Se comparan cuatro modalidades de XGBoost: entrenamiento centralizado
global, entrenamiento federado global mediante boosting cíclico,
entrenamiento centralizado por clúster predicho y entrenamiento federado
por clúster predicho. Los hiperparámetros y el número de clústeres se
seleccionan únicamente con validación. Test se evalúa después de fijar
cada ganador.


## 2. Importaciones y configuración del entorno


In [1]:
from pathlib import Path
from dataclasses import asdict, dataclass
from itertools import product
from typing import Dict, List, Mapping, Optional, Sequence, Tuple
import copy
import json
import math
import random
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.neighbors import (
    KNeighborsClassifier,
    KNeighborsRegressor,
    NearestNeighbors,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print


# Si Jupyter se inicia fuera de la carpeta del paquete, escribe aquí su ruta.
# Ejemplo: PROJECT_ROOT = Path(r"C:/TFM/notebooks_autocontenidos_sin_core")
PROJECT_ROOT = None

cwd = Path.cwd().resolve()
root_candidates = [cwd, cwd.parent, cwd.parent.parent]
if PROJECT_ROOT is not None:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    ROOT = next(
        (
            candidate
            for candidate in root_candidates
            if sum((candidate / name).is_dir() for name in ["TUT", "TUJI1", "UJIIndoor", "SOD"])
            >= 2
        ),
        cwd,
    )

SEED = 42
TARGET_COLUMNS = ["TARGET_X_M", "TARGET_Y_M"]

print("Raíz utilizada:", ROOT)


Raíz utilizada: /home/coder/Indoor/Notebooks


### 2.1. Carga y preprocesado

Estas funciones están dentro del notebook. Detectan las columnas
RSSI, cargan las particiones creadas por el notebook `00` y aplican
una transformación ajustada exclusivamente con `train`.


In [2]:
def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def natural_key(text: str) -> List[object]:
    return [int(piece) if piece.isdigit() else piece for piece in re.split(r"(\d+)", text)]


def detect_rssi_columns(df: pd.DataFrame) -> List[str]:
    cols = [c for c in df.columns if re.fullmatch(r"(?:WAP|MAC)\d+", str(c).upper())]
    cols = sorted(cols, key=natural_key)
    if not cols:
        raise ValueError("No se detectaron columnas RSSI WAPnnn o MACnnn.")
    return cols


class RSSIPreprocessor:
    """Imputa ausencias, estandariza RSSI con train y anade mascara de deteccion."""

    def __init__(self, missing_value: float = 100.0, fill_value: float = -110.0, use_mask: bool = True):
        self.missing_value = float(missing_value)
        self.fill_value = float(fill_value)
        self.use_mask = bool(use_mask)
        self.scaler = StandardScaler()
        self.columns: List[str] = []

    def _clean(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        raw = df[self.columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        observed = np.isfinite(raw) & (raw != self.missing_value)
        clean = np.where(observed, raw, self.fill_value).astype(np.float32)
        return clean, observed.astype(np.float32)

    def fit(self, df: pd.DataFrame, columns: Sequence[str]) -> "RSSIPreprocessor":
        self.columns = list(columns)
        clean, _ = self._clean(df)
        self.scaler.fit(clean)
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        clean, mask = self._clean(df)
        scaled = self.scaler.transform(clean).astype(np.float32)
        if self.use_mask:
            return np.concatenate([scaled, mask], axis=1).astype(np.float32)
        return scaled

    def fit_transform(self, df: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
        return self.fit(df, columns).transform(df)


#### Lectura de particiones y rutas


In [3]:
def read_base_splits(output_dir: Path, prefix: str) -> Dict[str, pd.DataFrame]:
    output_dir = Path(output_dir)
    return {
        split: pd.read_csv(output_dir / f"{prefix}_{split}.csv")
        for split in ["train", "val", "test"]
    }


def load_prepared_bundle(prepared_dir: Path, prefix: str) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    prepared_dir = Path(prepared_dir)
    splits = read_base_splits(prepared_dir, prefix)
    routes = pd.read_csv(prepared_dir / f"{prefix}_routes.csv")
    expected = {"ROW_ID", "SPLIT", "N_CLUSTERS", "CLUSTER", "CLUSTER_ORACLE"}
    if not expected.issubset(routes.columns):
        raise ValueError(f"El fichero de rutas no contiene {sorted(expected)}")
    return splits, routes


def route_frame(frame: pd.DataFrame, routes: pd.DataFrame, split: str, n_clusters: int) -> pd.DataFrame:
    selected = routes[
        (routes["SPLIT"].astype(str) == split)
        & (pd.to_numeric(routes["N_CLUSTERS"]) == int(n_clusters))
    ].copy()
    out = frame.merge(selected, on="ROW_ID", how="left", validate="one_to_one")
    if out["CLUSTER"].isna().any():
        raise ValueError(f"Faltan rutas para {split}, K={n_clusters}.")
    out["CLUSTER"] = out["CLUSTER"].astype(int)
    out["CLUSTER_ORACLE"] = out["CLUSTER_ORACLE"].astype(int)
    return out


### 2.2. Métricas y enrutamiento por clúster

El error de coordenadas es la distancia radial 2D en metros. Para
los modelos por zona, `CLUSTER` es la salida del router RSSI;
`CLUSTER_ORACLE` solo se utiliza como diagnóstico.


In [4]:
def coordinate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    truth = np.asarray(y_true, dtype=float)
    pred = np.asarray(y_pred, dtype=float)
    if truth.shape != pred.shape or truth.ndim != 2 or truth.shape[1] != 2:
        raise ValueError(f"Se esperaban matrices (n,2); recibidas {truth.shape} y {pred.shape}.")
    distances = np.linalg.norm(pred - truth, axis=1)
    return {
        "rmse_2d_m": float(np.sqrt(np.mean(distances ** 2))),
        "mean_error_m": float(np.mean(distances)),
        "median_error_m": float(np.median(distances)),
        "p75_error_m": float(np.quantile(distances, 0.75)),
        "p95_error_m": float(np.quantile(distances, 0.95)),
        "n": int(len(distances)),
    }


def _metric_row(
    model: str,
    modality: str,
    split: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_clusters: Optional[int] = None,
    extra: Optional[Mapping[str, object]] = None,
) -> Dict[str, object]:
    row: Dict[str, object] = {
        "model": model,
        "modality": modality,
        "split": split,
        "n_clusters": n_clusters,
        **coordinate_metrics(y_true, y_pred),
    }
    if extra:
        row.update(dict(extra))
    return row


def _predict_by_zone(
    models: Mapping[int, object],
    fallback: object,
    x: np.ndarray,
    zone_labels: np.ndarray,
) -> Tuple[np.ndarray, float]:
    pred = np.empty((len(x), 2), dtype=float)
    covered = np.zeros(len(x), dtype=bool)
    zones = np.asarray(zone_labels, dtype=int)
    for zone in np.unique(zones):
        mask = zones == zone
        model = models.get(int(zone), fallback)
        pred[mask] = model.predict(x[mask])
        covered[mask] = int(zone) in models
    return pred, float(np.mean(covered))


def compact_results(result: pd.DataFrame, split: str = "test") -> pd.DataFrame:
    columns = [
        "model",
        "modality",
        "n_clusters",
        "rmse_2d_m",
        "mean_error_m",
        "median_error_m",
        "p75_error_m",
        "p95_error_m",
        "coverage",
        "gate_accuracy_diagnostic",
    ]
    available = [c for c in columns if c in result.columns]
    return result[result["split"] == split][available].sort_values("rmse_2d_m").reset_index(drop=True)


## 3. Ficheros y configuración del dataset


In [5]:
DATASET = 'TUT'
MODEL_FAMILY = 'XGBoost'
PREPARED_DIR = ROOT / "prepared" / 'TUT'
PREFIX = 'tut_top5'
RESULTS_DIR = ROOT / "results" / 'TUT'
CLUSTER_VALUES = [2, 3, 4, 5, 6, 7, 8]
FLOOR_TASK = True
REQUIRE_KNOWN_TEST_CLIENTS = False

COORD_RESULTS_PATH = RESULTS_DIR / 'tut_top5_xgboost_coordinates_corrected.csv'
COORD_HPARAM_TUNING_PATH = COORD_RESULTS_PATH.with_name(
    COORD_RESULTS_PATH.stem + "_hyperparameter_tuning.csv"
)
COORD_CLUSTER_TUNING_PATH = COORD_RESULTS_PATH.with_name(
    COORD_RESULTS_PATH.stem + "_cluster_tuning.csv"
)
COORD_SELECTED_PATH = COORD_RESULTS_PATH.with_name(
    COORD_RESULTS_PATH.stem + "_selected_hyperparameters.json"
)

if FLOOR_TASK:
    FLOOR_RESULTS_PATH = RESULTS_DIR / 'tut_top5_xgboost_floor_corrected.csv'
    FLOOR_HPARAM_TUNING_PATH = FLOOR_RESULTS_PATH.with_name(
        FLOOR_RESULTS_PATH.stem + "_hyperparameter_tuning.csv"
    )
    FLOOR_CLUSTER_TUNING_PATH = FLOOR_RESULTS_PATH.with_name(
        FLOOR_RESULTS_PATH.stem + "_cluster_tuning.csv"
    )
    FLOOR_SELECTED_PATH = FLOOR_RESULTS_PATH.with_name(
        FLOOR_RESULTS_PATH.stem + "_selected_hyperparameters.json"
    )


In [6]:
required_inputs = [
    PREPARED_DIR / f"{PREFIX}_train.csv",
    PREPARED_DIR / f"{PREFIX}_val.csv",
    PREPARED_DIR / f"{PREFIX}_test.csv",
    PREPARED_DIR / f"{PREFIX}_routes.csv",
]
missing_inputs = [path for path in required_inputs if not path.exists()]

if missing_inputs:
    formatted = "\n- ".join(str(path) for path in missing_inputs)
    raise FileNotFoundError(
        "Faltan las particiones preparadas:\n- " + formatted
        + "\nEjecuta primero el notebook 00 del dataset incluido en este paquete."
    )

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Raíz del proyecto:", ROOT)
print("Datos preparados:", PREPARED_DIR)
print("Resultados:", RESULTS_DIR)


Raíz del proyecto: /home/coder/Indoor/Notebooks
Datos preparados: /home/coder/Indoor/Notebooks/prepared/TUT
Resultados: /home/coder/Indoor/Notebooks/results/TUT


## 4. Carga de datos

Las particiones y las rutas RSSI→clúster proceden del notebook `00` del
dataset. Validación y test no reutilizan clústeres calculados a partir de
sus coordenadas reales.


In [7]:
splits, routes = load_prepared_bundle(PREPARED_DIR, PREFIX)

position_columns = ["TARGET_X_M", "TARGET_Y_M"]
if "FLOOR_LABEL" in splits["train"].columns:
    position_columns.append("FLOOR_LABEL")

partition_summary = []
for split_name, frame in splits.items():
    partition_summary.append(
        {
            "split": split_name,
            "rows": len(frame),
            "positions": frame[position_columns].drop_duplicates().shape[0],
            "clients": frame["CLIENT_ID"].astype(str).nunique(),
            "floors": (
                frame["FLOOR_LABEL"].nunique()
                if "FLOOR_LABEL" in frame.columns
                else np.nan
            ),
        }
    )

display(pd.DataFrame(partition_summary).set_index("split"))
print("Valores de K disponibles:", sorted(routes["N_CLUSTERS"].astype(int).unique()))


,rows,positions,clients,floors
split,,,,
train,1868,1839,5,5
val,409,394,5,5
test,400,395,5,5


Valores de K disponibles: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


## 5. Auditoría de las particiones


In [8]:
def _row_ids(frame):
    return set(frame["ROW_ID"].astype(str))


def _positions(frame):
    return set(map(tuple, frame[position_columns].astype(str).to_numpy()))


audit_rows = []
for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
    audit_rows.append(
        {
            "partitions": f"{left}-{right}",
            "shared_row_ids": len(_row_ids(splits[left]) & _row_ids(splits[right])),
            "shared_positions": len(_positions(splits[left]) & _positions(splits[right])),
        }
    )

display(pd.DataFrame(audit_rows).set_index("partitions"))

train_clients = set(splits["train"]["CLIENT_ID"].astype(str))
val_clients = set(splits["val"]["CLIENT_ID"].astype(str))
test_clients = set(splits["test"]["CLIENT_ID"].astype(str))
unseen_vs_train = sorted(test_clients - train_clients)
unseen_vs_train_val = sorted(test_clients - (train_clients | val_clients))

print("Clientes de test ausentes en train:", unseen_vs_train)
print("Clientes de test ausentes en train/validación:", unseen_vs_train_val)

assert len(_row_ids(splits["train"]) & _row_ids(splits["val"])) == 0
assert len(_positions(splits["train"]) & _positions(splits["val"])) == 0
if REQUIRE_KNOWN_TEST_CLIENTS:
    assert unseen_vs_train_val == [], (
        "UJIIndoorLoc contiene dispositivos de test no presentes en train/validación: "
        f"{unseen_vs_train_val}. Vuelve a ejecutar el notebook 00 corregido."
    )

print("Auditoría superada.")


,shared_row_ids,shared_positions
partitions,,
train-val,0,0
train-test,0,0
val-test,0,0


Clientes de test ausentes en train: []
Clientes de test ausentes en train/validación: []
Auditoría superada.


## 6. Variables disponibles


In [9]:
rssi_columns = detect_rssi_columns(splits["train"])

variable_summary = pd.DataFrame(
    {
        "group": ["RSSI", "target", "client", "routing"],
        "columns": [
            len(rssi_columns),
            len(
                [
                    column
                    for column in ["TARGET_X_M", "TARGET_Y_M", "FLOOR_LABEL"]
                    if column in splits["train"]
                ]
            ),
            1,
            len(
                [
                    column
                    for column in ["ROW_ID", "SPLIT"]
                    if column in splits["train"]
                ]
            ),
        ],
    }
)
display(variable_summary.set_index("group"))
print("Primeras columnas RSSI:", rssi_columns[:10])
print(
    "Rango RSSI en train:",
    float(splits["train"][rssi_columns].min().min()),
    "a",
    float(splits["train"][rssi_columns].max().max()),
)


,columns
group,
RSSI,992
target,3
client,1
routing,1


Primeras columnas RSSI: ['WAP001', 'WAP002', 'WAP003', 'WAP004', 'WAP005', 'WAP006', 'WAP007', 'WAP008', 'WAP009', 'WAP010']
Rango RSSI en train: -102.0 a 100.0


## 7. Variables objetivo y métricas

La regresión usa `TARGET_X_M` y `TARGET_Y_M`. La selección se realiza con
RMSE radial 2D en metros y también se guardan error medio, mediana, P75 y
P95. Cuando existe `FLOOR_LABEL`, la clasificación de planta constituye
una tarea independiente y se selecciona mediante accuracy.


## 8. Preprocesado RSSI


In [10]:
# Esta celda documenta las dimensiones. Las funciones de entrenamiento vuelven
# a ajustar internamente un preprocesador idéntico usando exclusivamente train.
preprocessor_preview = RSSIPreprocessor(use_mask=True).fit(splits["train"], rssi_columns)
feature_shapes = {
    split_name: preprocessor_preview.transform(frame).shape
    for split_name, frame in splits.items()
}
display(
    pd.DataFrame(
        [
            {"split": name, "samples": shape[0], "features_after_mask": shape[1]}
            for name, shape in feature_shapes.items()
        ]
    ).set_index("split")
)
print("La imputación y el escalador RSSI se ajustan únicamente con train.")


,samples,features_after_mask
split,,
train,1868,1984
val,409,1984
test,400,1984


La imputación y el escalador RSSI se ajustan únicamente con train.


## 9. Modelo XGBoost y estrategia federada

Para coordenadas se ajustan dos boosters, uno para `TARGET_X_M` y otro
para `TARGET_Y_M`. Las coordenadas objetivo se estandarizan con train y
las predicciones se devuelven a metros antes de calcular las métricas.
Para planta se usa un booster multiclase.

La modalidad distribuida usa boosting cíclico: los clientes reciben el
ensemble compartido y añaden árboles secuencialmente con sus datos
locales. No se promedian predicciones ni modelos independientes. La mejor
iteración se conserva según validación; test no interviene.


### 9.1. Implementación autocontenida de XGBoost

Aquí se definen el modelo central, su estrategia distribuida o
federada y el procedimiento completo de selección con validación.
No se importa código propio desde ningún fichero `.py`.


#### Configuración y predictor XGBoost


In [11]:
@dataclass
class XGBConfig:
    max_depth: int = 8
    learning_rate: float = 0.05
    subsample: float = 0.85
    colsample_bytree: float = 0.55
    min_child_weight: float = 2.0
    reg_lambda: float = 1.0
    central_rounds: int = 400
    early_stopping_rounds: int = 30
    federated_cycles: int = 12
    local_boost_rounds: int = 1
    nthread: int = -1
    seed: int = SEED


class XGBCoordRegressor:
    def __init__(self, boosters, target_scaler: StandardScaler):
        self.boosters = boosters
        self.target_scaler = target_scaler

    def predict(self, x: np.ndarray) -> np.ndarray:
        import xgboost as xgb

        dmatrix = xgb.DMatrix(x)
        predictions = []
        for booster in self.boosters:
            best_iteration = getattr(booster, "best_iteration", None)
            if best_iteration is None:
                predictions.append(booster.predict(dmatrix))
            else:
                predictions.append(
                    booster.predict(dmatrix, iteration_range=(0, int(best_iteration) + 1))
                )
        scaled = np.column_stack(predictions)
        return self.target_scaler.inverse_transform(scaled)


def _xgb_params(config: XGBConfig) -> Dict[str, object]:
    return {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "max_depth": config.max_depth,
        "eta": config.learning_rate,
        "subsample": config.subsample,
        "colsample_bytree": config.colsample_bytree,
        "min_child_weight": config.min_child_weight,
        "lambda": config.reg_lambda,
        "nthread": config.nthread,
        "seed": config.seed,
    }


#### Entrenamiento XGBoost centralizado


In [12]:
def _fit_xgb_central(x_train, y_train_scaled, x_val, y_val_scaled, target_scaler, config):
    import xgboost as xgb

    boosters = []
    params = _xgb_params(config)
    for axis in range(2):
        dtrain = xgb.DMatrix(x_train, label=y_train_scaled[:, axis])
        dval = xgb.DMatrix(x_val, label=y_val_scaled[:, axis])
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=config.central_rounds,
            evals=[(dval, "val")],
            early_stopping_rounds=config.early_stopping_rounds,
            verbose_eval=False,
        )
        boosters.append(booster)
    return XGBCoordRegressor(boosters, target_scaler)


#### Boosting cíclico entre clientes


In [13]:
def _fit_xgb_cyclic(
    x_train,
    y_train_scaled,
    clients,
    x_val,
    y_val_scaled,
    target_scaler,
    config,
):
    """Boosting secuencial entre clientes; no promedia predicciones locales."""
    import xgboost as xgb

    params = _xgb_params(config)
    ids = np.asarray(clients).astype(str)
    unique = sorted(np.unique(ids))
    rng = np.random.default_rng(config.seed)
    dval = xgb.DMatrix(x_val)
    boosters = [None, None]
    best_boosters = [None, None]
    best_score = float("inf")
    for _cycle in range(config.federated_cycles):
        order = list(unique)
        rng.shuffle(order)
        for client in order:
            mask = ids == client
            if not np.any(mask):
                continue
            for axis in range(2):
                dclient = xgb.DMatrix(x_train[mask], label=y_train_scaled[mask, axis])
                boosters[axis] = xgb.train(
                    params,
                    dclient,
                    num_boost_round=config.local_boost_rounds,
                    xgb_model=boosters[axis],
                    verbose_eval=False,
                )
        pred = np.column_stack([booster.predict(dval) for booster in boosters])
        score = float(np.sqrt(np.mean(np.sum((pred - y_val_scaled) ** 2, axis=1))))
        if score < best_score:
            best_score = score
            best_boosters = [booster.copy() for booster in boosters]
    return XGBCoordRegressor(best_boosters, target_scaler)


#### Selección y evaluación XGBoost


In [14]:
def run_xgb_experiment(
    prepared_dir: Path,
    prefix: str,
    config: Optional[XGBConfig] = None,
    central_configs: Optional[Sequence[XGBConfig]] = None,
    federated_configs: Optional[Sequence[XGBConfig]] = None,
    cluster_values: Optional[Sequence[int]] = None,
    output_csv: Optional[Path] = None,
) -> pd.DataFrame:
    fallback_config = config or XGBConfig()
    central_candidates = (
        list(central_configs) if central_configs is not None else [fallback_config]
    )
    federated_candidates = (
        list(federated_configs) if federated_configs is not None else [fallback_config]
    )
    if not central_candidates or not federated_candidates:
        raise ValueError("La búsqueda XGBoost necesita al menos una configuración por modalidad.")
    seed_everything(fallback_config.seed)
    splits, routes = load_prepared_bundle(prepared_dir, prefix)
    rssi = detect_rssi_columns(splits["train"])
    pre = RSSIPreprocessor(use_mask=True).fit(splits["train"], rssi)
    x = {name: pre.transform(frame) for name, frame in splits.items()}
    y = {name: frame[TARGET_COLUMNS].to_numpy(dtype=float) for name, frame in splits.items()}
    target_scaler = StandardScaler().fit(y["train"])
    ys = {name: target_scaler.transform(values) for name, values in y.items()}
    clients = splits["train"]["CLIENT_ID"].astype(str).to_numpy()

    tuning_rows: List[Dict[str, object]] = []
    central_fitted: Dict[str, object] = {}
    federated_fitted: Dict[str, object] = {}
    central_by_id: Dict[str, XGBConfig] = {}
    federated_by_id: Dict[str, XGBConfig] = {}

    for index, candidate in enumerate(central_candidates):
        candidate_id = f"central_{index:02d}"
        fitted = _fit_xgb_central(
            x["train"], ys["train"], x["val"], ys["val"], target_scaler, candidate
        )
        central_fitted[candidate_id] = fitted
        central_by_id[candidate_id] = candidate
        tuning_rows.append(
            {
                "model": "XGBoost",
                "task": "coordinates",
                "selection_modality": "centralized_global",
                "candidate_id": candidate_id,
                "selection_split": "val",
                "selection_metric": "rmse_2d_m",
                **coordinate_metrics(y["val"], fitted.predict(x["val"])),
                **asdict(candidate),
            }
        )

    for index, candidate in enumerate(federated_candidates):
        candidate_id = f"federated_{index:02d}"
        fitted = _fit_xgb_cyclic(
            x["train"], ys["train"], clients, x["val"], ys["val"],
            target_scaler, candidate
        )
        federated_fitted[candidate_id] = fitted
        federated_by_id[candidate_id] = candidate
        tuning_rows.append(
            {
                "model": "XGBoost",
                "task": "coordinates",
                "selection_modality": "federated_global_cyclic_boosting",
                "candidate_id": candidate_id,
                "selection_split": "val",
                "selection_metric": "rmse_2d_m",
                **coordinate_metrics(y["val"], fitted.predict(x["val"])),
                **asdict(candidate),
            }
        )

    tuning_df = pd.DataFrame(tuning_rows)
    central_winner = (
        tuning_df[tuning_df["selection_modality"] == "centralized_global"]
        .sort_values(["rmse_2d_m", "candidate_id"])
        .iloc[0]
    )
    federated_winner = (
        tuning_df[
            tuning_df["selection_modality"] == "federated_global_cyclic_boosting"
        ]
        .sort_values(["rmse_2d_m", "candidate_id"])
        .iloc[0]
    )
    central_id = str(central_winner["candidate_id"])
    federated_id = str(federated_winner["candidate_id"])
    tuning_df["selected"] = (
        ((tuning_df["selection_modality"] == "centralized_global")
         & (tuning_df["candidate_id"] == central_id))
        | ((tuning_df["selection_modality"] == "federated_global_cyclic_boosting")
           & (tuning_df["candidate_id"] == federated_id))
    )
    central_config = central_by_id[central_id]
    federated_config = federated_by_id[federated_id]
    central_global = central_fitted[central_id]
    fed_global = federated_fitted[federated_id]
    del central_fitted, federated_fitted
    central_extra = {
        "selected_candidate_id": central_id,
        "selected_from_n_candidates": len(central_candidates),
        "selection_metric": "val_rmse_2d_m",
        **asdict(central_config),
    }
    federated_extra = {
        "selected_candidate_id": federated_id,
        "selected_from_n_candidates": len(federated_candidates),
        "selection_metric": "val_rmse_2d_m",
        **asdict(federated_config),
    }
    rows: List[Dict[str, object]] = []
    for split in ["val", "test"]:
        rows.extend(
            [
                _metric_row(
                    "XGBoost", "centralized_global", split, y[split], central_global.predict(x[split]),
                    extra=central_extra,
                ),
                _metric_row(
                    "XGBoost", "federated_global_cyclic_boosting", split, y[split], fed_global.predict(x[split]),
                    extra=federated_extra,
                ),
            ]
        )

    available = sorted(pd.to_numeric(routes["N_CLUSTERS"]).astype(int).unique())
    cluster_values = list(cluster_values) if cluster_values is not None else available
    cluster_cache: Dict[int, Tuple[Dict[int, object], Dict[int, object]]] = {}
    cluster_val_rows: List[Dict[str, object]] = []
    for k_clusters in [int(k) for k in cluster_values if int(k) in available]:
        routed = {
            split: route_frame(splits[split], routes, split, k_clusters)
            for split in ["train", "val"]
        }
        c_models: Dict[int, object] = {}
        f_models: Dict[int, object] = {}
        for zone in sorted(routed["train"]["CLUSTER"].unique()):
            train_mask = routed["train"]["CLUSTER"].to_numpy() == zone
            val_mask = routed["val"]["CLUSTER"].to_numpy() == zone
            if np.sum(train_mask) < 20 or np.sum(val_mask) < 3:
                continue
            c_models[int(zone)] = _fit_xgb_central(
                x["train"][train_mask], ys["train"][train_mask],
                x["val"][val_mask], ys["val"][val_mask], target_scaler, central_config
            )
            f_models[int(zone)] = _fit_xgb_cyclic(
                x["train"][train_mask], ys["train"][train_mask], clients[train_mask],
                x["val"][val_mask], ys["val"][val_mask], target_scaler, federated_config
            )
        cluster_cache[k_clusters] = (c_models, f_models)
        labels = routed["val"]["CLUSTER"].to_numpy(dtype=int)
        cpred, ccov = _predict_by_zone(c_models, central_global, x["val"], labels)
        fpred, fcov = _predict_by_zone(f_models, fed_global, x["val"], labels)
        gate_acc = float(np.mean(labels == routed["val"]["CLUSTER_ORACLE"].to_numpy()))
        cluster_val_rows.extend(
            [
                _metric_row(
                    "XGBoost", "centralized_by_predicted_cluster", "val", y["val"], cpred,
                    n_clusters=k_clusters,
                    extra={"coverage": ccov, "gate_accuracy_diagnostic": gate_acc, **central_extra},
                ),
                _metric_row(
                    "XGBoost", "federated_by_predicted_cluster_cyclic_boosting", "val",
                    y["val"], fpred, n_clusters=k_clusters,
                    extra={"coverage": fcov, "gate_accuracy_diagnostic": gate_acc, **federated_extra},
                ),
            ]
        )
    cluster_tuning_df = pd.DataFrame(cluster_val_rows)
    for modality in [
        "centralized_by_predicted_cluster",
        "federated_by_predicted_cluster_cyclic_boosting",
    ]:
        val_rows = cluster_tuning_df[cluster_tuning_df["modality"] == modality]
        if not val_rows.empty:
            best_k = int(val_rows.sort_values("rmse_2d_m").iloc[0]["n_clusters"])
            rows.append(val_rows[val_rows["n_clusters"] == best_k].iloc[0].to_dict())
            test_routed = route_frame(splits["test"], routes, "test", best_k)
            labels = test_routed["CLUSTER"].to_numpy(dtype=int)
            c_models, f_models = cluster_cache[best_k]
            if modality == "centralized_by_predicted_cluster":
                pred, coverage = _predict_by_zone(c_models, central_global, x["test"], labels)
            else:
                pred, coverage = _predict_by_zone(f_models, fed_global, x["test"], labels)
            rows.append(
                _metric_row(
                    "XGBoost", modality, "test", y["test"], pred, n_clusters=best_k,
                    extra={
                        "coverage": coverage,
                        "gate_accuracy_diagnostic": float(
                            np.mean(labels == test_routed["CLUSTER_ORACLE"].to_numpy())
                        ),
                        **(
                            central_extra
                            if modality == "centralized_by_predicted_cluster"
                            else federated_extra
                        ),
                    },
                )
            )
    keep = pd.DataFrame(rows).sort_values(["split", "modality"]).reset_index(drop=True)
    if output_csv is not None:
        Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
        keep.to_csv(output_csv, index=False)
        tuning_df.to_csv(
            Path(output_csv).with_name(
                Path(output_csv).stem + "_hyperparameter_tuning.csv"
            ),
            index=False,
        )
        cluster_tuning_df.to_csv(
            Path(output_csv).with_name(Path(output_csv).stem + "_cluster_tuning.csv"), index=False
        )
        selection_path = Path(output_csv).with_name(
            Path(output_csv).stem + "_selected_hyperparameters.json"
        )
        with selection_path.open("w", encoding="utf-8") as handle:
            json.dump(
                {
                    "model": "XGBoost",
                    "task": "coordinates",
                    "selection_split": "validation",
                    "selection_metric": "rmse_2d_m",
                    "centralized_global": {
                        "candidate_id": central_id,
                        **asdict(central_config),
                    },
                    "federated_global_cyclic_boosting": {
                        "candidate_id": federated_id,
                        **asdict(federated_config),
                    },
                    "cluster_protocol": (
                        "The centralized and federated winners are reused in their "
                        "respective cluster-aware strategies; K is selected on validation."
                    ),
                },
                handle,
                indent=2,
                ensure_ascii=False,
            )
    return keep


## 10. Espacio de hiperparámetros


In [15]:
COMMON = dict(
    subsample=0.85,
    min_child_weight=2.0,
    reg_lambda=1.0,
    central_rounds=400,
    early_stopping_rounds=30,
    federated_cycles=12,
    nthread=-1,
    seed=42,
)

CENTRAL_CONFIGS = [
    XGBConfig(
        max_depth=max_depth,
        learning_rate=learning_rate,
        colsample_bytree=colsample,
        local_boost_rounds=1,
        **COMMON,
    )
    for max_depth, learning_rate, colsample in product(
        (4, 8),
        (0.05, 0.10),
        (0.55, 0.80),
    )
]

FEDERATED_CONFIGS = [
    XGBConfig(
        max_depth=max_depth,
        learning_rate=learning_rate,
        colsample_bytree=0.55,
        local_boost_rounds=local_rounds,
        **COMMON,
    )
    for max_depth, learning_rate, local_rounds in product(
        (4, 8),
        (0.05, 0.10),
        (1, 2),
    )
]

TUNING_DISPLAY_COLUMNS = [
    "selection_modality",
    "candidate_id",
    "selected",
    "rmse_2d_m",
    "mean_error_m",
    "max_depth",
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "min_child_weight",
    "reg_lambda",
    "central_rounds",
    "federated_cycles",
    "local_boost_rounds",
]
FLOOR_TUNING_DISPLAY_COLUMNS = [
    "selection_modality",
    "candidate_id",
    "selected",
    "floor_accuracy",
    "max_depth",
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "min_child_weight",
    "reg_lambda",
    "central_rounds",
    "federated_cycles",
    "local_boost_rounds",
]

print(
    "Candidatos XGBoost:",
    len(CENTRAL_CONFIGS),
    "centrales y",
    len(FEDERATED_CONFIGS),
    "federados",
)


Candidatos XGBoost: 8 centrales y 8 federados


In [16]:
central_grid = pd.DataFrame(
    [{"candidate_id": f"central_{index:02d}", **asdict(config)} for index, config in enumerate(CENTRAL_CONFIGS)]
)
federated_grid = pd.DataFrame(
    [{"candidate_id": f"federated_{index:02d}", **asdict(config)} for index, config in enumerate(FEDERATED_CONFIGS)]
)

print("=== Rejilla centralizada ===")
display(central_grid)
print("=== Rejilla federada ===")
display(federated_grid)
print("Valores de K espacial evaluados:", CLUSTER_VALUES)


=== Rejilla centralizada ===


,candidate_id,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,reg_lambda,central_rounds,early_stopping_rounds,federated_cycles,local_boost_rounds,nthread,seed
0,central_00,4,0.05,0.85,0.55,2.0,1.0,400,30,12,1,-1,42
1,central_01,4,0.05,0.85,0.80,2.0,1.0,400,30,12,1,-1,42
2,central_02,4,0.10,0.85,0.55,2.0,1.0,400,30,12,1,-1,42
3,central_03,4,0.10,0.85,0.80,2.0,1.0,400,30,12,1,-1,42
4,central_04,8,0.05,0.85,0.55,2.0,1.0,400,30,12,1,-1,42
5,central_05,8,0.05,0.85,0.80,2.0,1.0,400,30,12,1,-1,42
6,central_06,8,0.10,0.85,0.55,2.0,1.0,400,30,12,1,-1,42
7,central_07,8,0.10,0.85,0.80,2.0,1.0,400,30,12,1,-1,42


=== Rejilla federada ===


,candidate_id,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,reg_lambda,central_rounds,early_stopping_rounds,federated_cycles,local_boost_rounds,nthread,seed
0,federated_00,4,0.05,0.85,0.55,2.0,1.0,400,30,12,1,-1,42
1,federated_01,4,0.05,0.85,0.55,2.0,1.0,400,30,12,2,-1,42
2,federated_02,4,0.10,0.85,0.55,2.0,1.0,400,30,12,1,-1,42
3,federated_03,4,0.10,0.85,0.55,2.0,1.0,400,30,12,2,-1,42
4,federated_04,8,0.05,0.85,0.55,2.0,1.0,400,30,12,1,-1,42
5,federated_05,8,0.05,0.85,0.55,2.0,1.0,400,30,12,2,-1,42
6,federated_06,8,0.10,0.85,0.55,2.0,1.0,400,30,12,1,-1,42
7,federated_07,8,0.10,0.85,0.55,2.0,1.0,400,30,12,2,-1,42


Valores de K espacial evaluados: [2, 3, 4, 5, 6, 7, 8]


## 11. Estrategias de entrenamiento


### 11.1. Modelos globales

La modalidad centralizada ajusta el ensemble con todas las muestras de
train y aplica early stopping con validación. La modalidad federada
mantiene las muestras separadas por `CLIENT_ID` y añade árboles de forma
cíclica. Cada modalidad selecciona sus propios hiperparámetros.


### 11.2. Modelos por clúster

El ganador centralizado y el ganador de boosting cíclico se reutilizan en
sus respectivas estrategias por clúster. El número de clústeres se elige
por modalidad con validación. El router usa exclusivamente RSSI en
validación y test; `CLUSTER_ORACLE`, calculado geométricamente, solo se
muestra como diagnóstico.


## 12. Entrenamiento y evaluación de coordenadas


In [17]:
coordinate_results = run_xgb_experiment(
    prepared_dir=PREPARED_DIR,
    prefix=PREFIX,
    central_configs=CENTRAL_CONFIGS,
    federated_configs=FEDERATED_CONFIGS,
    cluster_values=CLUSTER_VALUES,
    output_csv=COORD_RESULTS_PATH,
)
print("Experimento de coordenadas finalizado.")


Experimento de coordenadas finalizado.


### 12.1. Búsqueda global con validación


In [18]:
coordinate_tuning = pd.read_csv(COORD_HPARAM_TUNING_PATH)
coordinate_tuning = coordinate_tuning.sort_values(
    ["selection_modality", "rmse_2d_m", "candidate_id"]
).reset_index(drop=True)

coordinate_tuning_columns = [
    column
    for column in TUNING_DISPLAY_COLUMNS
    if column in coordinate_tuning.columns
]
print("La selección global se realiza con RMSE radial 2D de validación.")
display(coordinate_tuning[coordinate_tuning_columns])


La selección global se realiza con RMSE radial 2D de validación.


,selection_modality,candidate_id,selected,rmse_2d_m,mean_error_m,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,reg_lambda,central_rounds,federated_cycles,local_boost_rounds
0,centralized_global,central_06,True,9.482886,6.393659,8,0.10,0.85,0.55,2.0,1.0,400,12,1
1,centralized_global,central_04,False,9.514378,6.364436,8,0.05,0.85,0.55,2.0,1.0,400,12,1
2,centralized_global,central_05,False,9.751138,6.532172,8,0.05,0.85,0.80,2.0,1.0,400,12,1
3,centralized_global,central_07,False,9.763720,6.581821,8,0.10,0.85,0.80,2.0,1.0,400,12,1
4,centralized_global,central_02,False,9.816262,7.252434,4,0.10,0.85,0.55,2.0,1.0,400,12,1
5,centralized_global,central_03,False,10.111579,7.419295,4,0.10,0.85,0.80,2.0,1.0,400,12,1
6,centralized_global,central_00,False,10.457535,7.694544,4,0.05,0.85,0.55,2.0,1.0,400,12,1
7,centralized_global,central_01,False,10.624816,7.881570,4,0.05,0.85,0.80,2.0,1.0,400,12,1
8,federated_global_cyclic_boosting,federated_07,True,13.303742,10.114700,8,0.10,0.85,0.55,2.0,1.0,400,12,2
9,federated_global_cyclic_boosting,federated_06,False,13.653121,10.240348,8,0.10,0.85,0.55,2.0,1.0,400,12,1


### 12.2. Selección del número de clústeres


In [19]:
coordinate_cluster_tuning = pd.read_csv(COORD_CLUSTER_TUNING_PATH)
coordinate_cluster_tuning = coordinate_cluster_tuning.sort_values(
    ["modality", "rmse_2d_m", "n_clusters"]
).reset_index(drop=True)

display(
    coordinate_cluster_tuning[
        [
            "modality",
            "n_clusters",
            "rmse_2d_m",
            "mean_error_m",
            "coverage",
            "gate_accuracy_diagnostic",
            "selected_candidate_id",
        ]
    ]
)


,modality,n_clusters,rmse_2d_m,mean_error_m,coverage,gate_accuracy_diagnostic,selected_candidate_id
0,centralized_by_predicted_cluster,6,6.873063,4.958806,1.0,0.897311,central_06
1,centralized_by_predicted_cluster,5,6.954088,4.929232,1.0,0.897311,central_06
2,centralized_by_predicted_cluster,4,7.162153,4.998783,1.0,0.931540,central_06
3,centralized_by_predicted_cluster,7,7.179406,5.113040,1.0,0.882641,central_06
4,centralized_by_predicted_cluster,2,7.187672,5.168422,1.0,0.973105,central_06
5,centralized_by_predicted_cluster,3,7.282562,5.165943,1.0,0.933985,central_06
6,centralized_by_predicted_cluster,8,7.554225,5.178365,1.0,0.845966,central_06
7,federated_by_predicted_cluster_cyclic_boosting,7,8.435412,6.392376,1.0,0.882641,federated_07
8,federated_by_predicted_cluster_cyclic_boosting,8,8.987198,6.640629,1.0,0.845966,federated_07
9,federated_by_predicted_cluster_cyclic_boosting,5,9.516373,7.162892,1.0,0.897311,federated_07


### 12.3. Resultados de validación y test


In [20]:
print("=== Coordenadas: validación ===")
display(compact_results(coordinate_results, split="val"))

print("=== Coordenadas: test final ===")
display(compact_results(coordinate_results, split="test"))


=== Coordenadas: validación ===


,model,modality,n_clusters,rmse_2d_m,mean_error_m,median_error_m,p75_error_m,p95_error_m,coverage,gate_accuracy_diagnostic
0,XGBoost,centralized_by_predicted_cluster,6.0,6.873063,4.958806,3.683625,6.155643,12.412822,1.0,0.897311
1,XGBoost,federated_by_predicted_cluster_cyclic_boosting,7.0,8.435412,6.392376,5.296926,7.903983,14.005435,1.0,0.882641
2,XGBoost,centralized_global,NaN,9.482886,6.393659,4.764988,7.583012,15.904443,NaN,NaN
3,XGBoost,federated_global_cyclic_boosting,NaN,13.303742,10.114700,7.918296,12.337224,24.837186,NaN,NaN


=== Coordenadas: test final ===


,model,modality,n_clusters,rmse_2d_m,mean_error_m,median_error_m,p75_error_m,p95_error_m,coverage,gate_accuracy_diagnostic
0,XGBoost,centralized_by_predicted_cluster,6.0,8.357606,5.421491,3.985573,6.669059,12.554969,1.0,0.9075
1,XGBoost,federated_by_predicted_cluster_cyclic_boosting,7.0,8.608230,6.477052,5.112672,8.288602,13.821601,1.0,0.9025
2,XGBoost,centralized_global,NaN,9.490698,6.571708,4.686711,7.563651,19.283758,NaN,NaN
3,XGBoost,federated_global_cyclic_boosting,NaN,14.055876,10.817003,8.284776,13.773378,27.197928,NaN,NaN


### 12.4. Hiperparámetros seleccionados


In [21]:
with COORD_SELECTED_PATH.open("r", encoding="utf-8") as handle:
    selected_coordinate_hyperparameters = json.load(handle)

print(json.dumps(selected_coordinate_hyperparameters, indent=2, ensure_ascii=False))


{
  "model": "XGBoost",
  "task": "coordinates",
  "selection_split": "validation",
  "selection_metric": "rmse_2d_m",
  "centralized_global": {
    "candidate_id": "central_06",
    "max_depth": 8,
    "learning_rate": 0.1,
    "subsample": 0.85,
    "colsample_bytree": 0.55,
    "min_child_weight": 2.0,
    "reg_lambda": 1.0,
    "central_rounds": 400,
    "early_stopping_rounds": 30,
    "federated_cycles": 12,
    "local_boost_rounds": 1,
    "nthread": -1,
    "seed": 42
  },
  "federated_global_cyclic_boosting": {
    "candidate_id": "federated_07",
    "max_depth": 8,
    "learning_rate": 0.1,
    "subsample": 0.85,
    "colsample_bytree": 0.55,
    "min_child_weight": 2.0,
    "reg_lambda": 1.0,
    "central_rounds": 400,
    "early_stopping_rounds": 30,
    "federated_cycles": 12,
    "local_boost_rounds": 2,
    "nthread": -1,
    "seed": 42
  },
  "cluster_protocol": "The centralized and federated winners are reused in their respective cluster-aware strategies; K is selected

## 13. Clasificación de planta

La tarea de planta repite la búsqueda completa. Tanto los
hiperparámetros globales como el K espacial se eligen con accuracy
de validación, sin reutilizar la selección de coordenadas.


### Implementación de la clasificación de planta

La clasificación de planta repite la selección con accuracy de
validación y mantiene separadas las configuraciones de coordenadas
y planta.


#### Preparación, métricas y predicción por zona


In [22]:
def floor_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    truth = np.asarray(y_true, dtype=int)
    pred = np.asarray(y_pred, dtype=int)
    return {"floor_accuracy": float(np.mean(truth == pred)), "n": int(len(truth))}


def _floor_row(
    model: str,
    modality: str,
    split: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_clusters: Optional[int] = None,
    extra: Optional[Mapping[str, object]] = None,
) -> Dict[str, object]:
    row: Dict[str, object] = {
        "model": model,
        "task": "floor",
        "modality": modality,
        "split": split,
        "n_clusters": n_clusters,
        **floor_metrics(y_true, y_pred),
    }
    if extra:
        row.update(dict(extra))
    return row


def _floor_setup(prepared_dir: Path, prefix: str):
    splits, routes = load_prepared_bundle(prepared_dir, prefix)
    if "FLOOR_LABEL" not in splits["train"].columns:
        raise ValueError("El dataset preparado no contiene FLOOR_LABEL.")
    rssi = detect_rssi_columns(splits["train"])
    pre = RSSIPreprocessor(use_mask=True).fit(splits["train"], rssi)
    x = {split: pre.transform(frame) for split, frame in splits.items()}
    encoder = LabelEncoder().fit(splits["train"]["FLOOR_LABEL"])
    y = {
        split: encoder.transform(frame["FLOOR_LABEL"]).astype(int)
        for split, frame in splits.items()
    }
    clients = splits["train"]["CLIENT_ID"].astype(str).to_numpy()
    return splits, routes, x, y, clients, encoder


def _predict_floor_by_zone(
    models: Mapping[int, object],
    fallback: object,
    x: np.ndarray,
    labels: np.ndarray,
) -> Tuple[np.ndarray, float]:
    output = np.empty(len(x), dtype=int)
    covered = np.zeros(len(x), dtype=bool)
    for zone in np.unique(labels):
        mask = labels == zone
        model = models.get(int(zone), fallback)
        output[mask] = model.predict(x[mask])
        covered[mask] = int(zone) in models
    return output, float(np.mean(covered))


def compact_floor_results(result: pd.DataFrame, split: str = "test") -> pd.DataFrame:
    columns = [
        "model",
        "modality",
        "n_clusters",
        "floor_accuracy",
        "coverage",
        "gate_accuracy_diagnostic",
    ]
    available = [column for column in columns if column in result.columns]
    return (
        result[result["split"] == split][available]
        .sort_values("floor_accuracy", ascending=False)
        .reset_index(drop=True)
    )


#### Clasificador y parámetros XGBoost de planta


In [23]:
class XGBFloorClassifier:
    def __init__(self, booster):
        self.booster = booster

    def predict(self, x: np.ndarray) -> np.ndarray:
        import xgboost as xgb

        matrix = xgb.DMatrix(x)
        best_iteration = getattr(self.booster, "best_iteration", None)
        if best_iteration is None:
            probabilities = self.booster.predict(matrix)
        else:
            probabilities = self.booster.predict(
                matrix, iteration_range=(0, int(best_iteration) + 1)
            )
        if probabilities.ndim == 1:
            return probabilities.astype(int)
        return np.argmax(probabilities, axis=1).astype(int)


def _xgb_floor_parameters(config: XGBConfig, n_classes: int) -> Dict[str, object]:
    return {
        "objective": "multi:softprob",
        "eval_metric": "mlogloss",
        "num_class": int(n_classes),
        "tree_method": "hist",
        "max_depth": config.max_depth,
        "eta": config.learning_rate,
        "subsample": config.subsample,
        "colsample_bytree": config.colsample_bytree,
        "min_child_weight": config.min_child_weight,
        "lambda": config.reg_lambda,
        "nthread": config.nthread,
        "seed": config.seed,
    }


#### Entrenamiento central y cíclico de planta


In [24]:
def _fit_xgb_floor_central(x_train, y_train, x_val, y_val, n_classes, config):
    import xgboost as xgb

    booster = xgb.train(
        _xgb_floor_parameters(config, n_classes),
        xgb.DMatrix(x_train, label=y_train),
        num_boost_round=config.central_rounds,
        evals=[(xgb.DMatrix(x_val, label=y_val), "val")],
        early_stopping_rounds=config.early_stopping_rounds,
        verbose_eval=False,
    )
    return XGBFloorClassifier(booster)


def _fit_xgb_floor_cyclic(x_train, y_train, clients, x_val, y_val, n_classes, config):
    import xgboost as xgb

    params = _xgb_floor_parameters(config, n_classes)
    ids = np.asarray(clients).astype(str)
    rng = np.random.default_rng(config.seed)
    booster = None
    best_booster = None
    best_accuracy = -1.0
    for _cycle in range(config.federated_cycles):
        order = sorted(np.unique(ids))
        rng.shuffle(order)
        for client in order:
            mask = ids == client
            if not np.any(mask):
                continue
            booster = xgb.train(
                params,
                xgb.DMatrix(x_train[mask], label=y_train[mask]),
                num_boost_round=config.local_boost_rounds,
                xgb_model=booster,
                verbose_eval=False,
            )
        pred = XGBFloorClassifier(booster).predict(x_val)
        accuracy = floor_metrics(y_val, pred)["floor_accuracy"]
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_booster = booster.copy()
    return XGBFloorClassifier(best_booster)


#### Selección y evaluación XGBoost de planta


In [25]:
def run_xgb_floor_experiment(
    prepared_dir: Path,
    prefix: str,
    config: Optional[XGBConfig] = None,
    central_configs: Optional[Sequence[XGBConfig]] = None,
    federated_configs: Optional[Sequence[XGBConfig]] = None,
    cluster_values: Optional[Sequence[int]] = None,
    output_csv: Optional[Path] = None,
) -> pd.DataFrame:
    fallback_config = config or XGBConfig()
    central_candidates = (
        list(central_configs) if central_configs is not None else [fallback_config]
    )
    federated_candidates = (
        list(federated_configs) if federated_configs is not None else [fallback_config]
    )
    if not central_candidates or not federated_candidates:
        raise ValueError("La búsqueda XGBoost de planta necesita candidatos por modalidad.")
    seed_everything(fallback_config.seed)
    splits, routes, x, y, clients, encoder = _floor_setup(prepared_dir, prefix)
    n_classes = len(encoder.classes_)
    tuning_rows: List[Dict[str, object]] = []
    central_fitted: Dict[str, object] = {}
    federated_fitted: Dict[str, object] = {}
    central_by_id: Dict[str, XGBConfig] = {}
    federated_by_id: Dict[str, XGBConfig] = {}

    for index, candidate in enumerate(central_candidates):
        candidate_id = f"central_{index:02d}"
        fitted = _fit_xgb_floor_central(
            x["train"], y["train"], x["val"], y["val"], n_classes, candidate
        )
        central_fitted[candidate_id] = fitted
        central_by_id[candidate_id] = candidate
        tuning_rows.append(
            {
                "model": "XGBoost",
                "task": "floor",
                "selection_modality": "centralized_global",
                "candidate_id": candidate_id,
                "selection_split": "val",
                "selection_metric": "floor_accuracy",
                **floor_metrics(y["val"], fitted.predict(x["val"])),
                **asdict(candidate),
            }
        )

    for index, candidate in enumerate(federated_candidates):
        candidate_id = f"federated_{index:02d}"
        fitted = _fit_xgb_floor_cyclic(
            x["train"], y["train"], clients, x["val"], y["val"], n_classes,
            candidate
        )
        federated_fitted[candidate_id] = fitted
        federated_by_id[candidate_id] = candidate
        tuning_rows.append(
            {
                "model": "XGBoost",
                "task": "floor",
                "selection_modality": "federated_global_cyclic_boosting",
                "candidate_id": candidate_id,
                "selection_split": "val",
                "selection_metric": "floor_accuracy",
                **floor_metrics(y["val"], fitted.predict(x["val"])),
                **asdict(candidate),
            }
        )

    tuning_df = pd.DataFrame(tuning_rows)
    central_winner = (
        tuning_df[tuning_df["selection_modality"] == "centralized_global"]
        .sort_values(["floor_accuracy", "candidate_id"], ascending=[False, True])
        .iloc[0]
    )
    federated_winner = (
        tuning_df[
            tuning_df["selection_modality"] == "federated_global_cyclic_boosting"
        ]
        .sort_values(["floor_accuracy", "candidate_id"], ascending=[False, True])
        .iloc[0]
    )
    central_id = str(central_winner["candidate_id"])
    federated_id = str(federated_winner["candidate_id"])
    tuning_df["selected"] = (
        ((tuning_df["selection_modality"] == "centralized_global")
         & (tuning_df["candidate_id"] == central_id))
        | ((tuning_df["selection_modality"] == "federated_global_cyclic_boosting")
           & (tuning_df["candidate_id"] == federated_id))
    )
    central_config = central_by_id[central_id]
    federated_config = federated_by_id[federated_id]
    central_global = central_fitted[central_id]
    fed_global = federated_fitted[federated_id]
    del central_fitted, federated_fitted
    central_extra = {
        "selected_candidate_id": central_id,
        "selected_from_n_candidates": len(central_candidates),
        "selection_metric": "val_floor_accuracy",
        **asdict(central_config),
    }
    federated_extra = {
        "selected_candidate_id": federated_id,
        "selected_from_n_candidates": len(federated_candidates),
        "selection_metric": "val_floor_accuracy",
        **asdict(federated_config),
    }
    rows: List[Dict[str, object]] = []
    for split in ["val", "test"]:
        rows.extend(
            [
                _floor_row(
                    "XGBoost", "centralized_global", split, y[split],
                    central_global.predict(x[split]), extra=central_extra
                ),
                _floor_row(
                    "XGBoost", "federated_global_cyclic_boosting", split, y[split],
                    fed_global.predict(x[split]), extra=federated_extra
                ),
            ]
        )
    available = sorted(pd.to_numeric(routes["N_CLUSTERS"]).astype(int).unique())
    values = list(cluster_values) if cluster_values is not None else available
    cache: Dict[int, Tuple[Dict[int, object], Dict[int, object]]] = {}
    validation_rows: List[Dict[str, object]] = []
    for clusters in [int(k) for k in values if int(k) in available]:
        routed_train = route_frame(splits["train"], routes, "train", clusters)
        routed_val = route_frame(splits["val"], routes, "val", clusters)
        central_models: Dict[int, object] = {}
        fed_models: Dict[int, object] = {}
        for zone in sorted(routed_train["CLUSTER"].unique()):
            train_mask = routed_train["CLUSTER"].to_numpy() == zone
            val_mask = routed_val["CLUSTER"].to_numpy() == zone
            if np.sum(train_mask) < 20 or np.sum(val_mask) < 3:
                continue
            central_models[int(zone)] = _fit_xgb_floor_central(
                x["train"][train_mask], y["train"][train_mask],
                x["val"][val_mask], y["val"][val_mask], n_classes, central_config
            )
            fed_models[int(zone)] = _fit_xgb_floor_cyclic(
                x["train"][train_mask], y["train"][train_mask], clients[train_mask],
                x["val"][val_mask], y["val"][val_mask], n_classes, federated_config
            )
        cache[clusters] = (central_models, fed_models)
        labels = routed_val["CLUSTER"].to_numpy(dtype=int)
        cpred, ccov = _predict_floor_by_zone(central_models, central_global, x["val"], labels)
        fpred, fcov = _predict_floor_by_zone(fed_models, fed_global, x["val"], labels)
        gate_accuracy = float(np.mean(labels == routed_val["CLUSTER_ORACLE"].to_numpy()))
        validation_rows.extend(
            [
                _floor_row(
                    "XGBoost", "centralized_by_predicted_cluster", "val", y["val"], cpred,
                    clusters, {"coverage": ccov, "gate_accuracy_diagnostic": gate_accuracy, **central_extra}
                ),
                _floor_row(
                    "XGBoost", "federated_by_predicted_cluster_cyclic_boosting", "val",
                    y["val"], fpred, clusters,
                    {"coverage": fcov, "gate_accuracy_diagnostic": gate_accuracy, **federated_extra}
                ),
            ]
        )
    cluster_tuning = pd.DataFrame(validation_rows)
    for modality in [
        "centralized_by_predicted_cluster",
        "federated_by_predicted_cluster_cyclic_boosting",
    ]:
        candidates = cluster_tuning[cluster_tuning["modality"] == modality]
        if candidates.empty:
            continue
        winner = candidates.sort_values("floor_accuracy", ascending=False).iloc[0]
        best_clusters = int(winner["n_clusters"])
        rows.append(winner.to_dict())
        routed_test = route_frame(splits["test"], routes, "test", best_clusters)
        labels = routed_test["CLUSTER"].to_numpy(dtype=int)
        central_models, fed_models = cache[best_clusters]
        models, fallback = (
            (central_models, central_global)
            if modality == "centralized_by_predicted_cluster"
            else (fed_models, fed_global)
        )
        pred, coverage = _predict_floor_by_zone(models, fallback, x["test"], labels)
        rows.append(
            _floor_row(
                "XGBoost", modality, "test", y["test"], pred, best_clusters,
                {
                    "coverage": coverage,
                    "gate_accuracy_diagnostic": float(
                        np.mean(labels == routed_test["CLUSTER_ORACLE"].to_numpy())
                    ),
                    **(
                        central_extra
                        if modality == "centralized_by_predicted_cluster"
                        else federated_extra
                    ),
                },
            )
        )
    result = pd.DataFrame(rows).sort_values(["split", "modality"]).reset_index(drop=True)
    if output_csv is not None:
        path = Path(output_csv)
        path.parent.mkdir(parents=True, exist_ok=True)
        result.to_csv(path, index=False)
        tuning_df.to_csv(
            path.with_name(path.stem + "_hyperparameter_tuning.csv"), index=False
        )
        cluster_tuning.to_csv(path.with_name(path.stem + "_cluster_tuning.csv"), index=False)
        with path.with_name(path.stem + "_selected_hyperparameters.json").open(
            "w", encoding="utf-8"
        ) as handle:
            json.dump(
                {
                    "model": "XGBoost",
                    "task": "floor",
                    "selection_split": "validation",
                    "selection_metric": "floor_accuracy",
                    "centralized_global": {
                        "candidate_id": central_id,
                        **asdict(central_config),
                    },
                    "federated_global_cyclic_boosting": {
                        "candidate_id": federated_id,
                        **asdict(federated_config),
                    },
                    "cluster_protocol": (
                        "The centralized and federated winners are reused in their "
                        "respective cluster-aware strategies; K is selected on validation."
                    ),
                },
                handle,
                indent=2,
                ensure_ascii=False,
            )
    return result


In [26]:
# Planta repite la búsqueda y maximiza accuracy de validación. No reutiliza
# automáticamente el ganador obtenido para coordenadas.
floor_results = run_xgb_floor_experiment(
    prepared_dir=PREPARED_DIR,
    prefix=PREFIX,
    central_configs=CENTRAL_CONFIGS,
    federated_configs=FEDERATED_CONFIGS,
    cluster_values=CLUSTER_VALUES,
    output_csv=FLOOR_RESULTS_PATH,
)
print("Experimento de clasificación de planta finalizado.")


Experimento de clasificación de planta finalizado.


### 13.1. Búsqueda global con validación


In [27]:
floor_tuning = pd.read_csv(FLOOR_HPARAM_TUNING_PATH)
floor_tuning = floor_tuning.sort_values(
    ["selection_modality", "floor_accuracy", "candidate_id"],
    ascending=[True, False, True],
).reset_index(drop=True)

floor_tuning_columns = [
    column
    for column in FLOOR_TUNING_DISPLAY_COLUMNS
    if column in floor_tuning.columns
]
print("La selección global de planta se realiza con accuracy de validación.")
display(floor_tuning[floor_tuning_columns])


La selección global de planta se realiza con accuracy de validación.


,selection_modality,candidate_id,selected,floor_accuracy,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,reg_lambda,central_rounds,federated_cycles,local_boost_rounds
0,centralized_global,central_00,True,0.968215,4,0.05,0.85,0.55,2.0,1.0,400,12,1
1,centralized_global,central_02,False,0.965770,4,0.10,0.85,0.55,2.0,1.0,400,12,1
2,centralized_global,central_03,False,0.965770,4,0.10,0.85,0.80,2.0,1.0,400,12,1
3,centralized_global,central_04,False,0.963325,8,0.05,0.85,0.55,2.0,1.0,400,12,1
4,centralized_global,central_05,False,0.960880,8,0.05,0.85,0.80,2.0,1.0,400,12,1
5,centralized_global,central_06,False,0.960880,8,0.10,0.85,0.55,2.0,1.0,400,12,1
6,centralized_global,central_01,False,0.958435,4,0.05,0.85,0.80,2.0,1.0,400,12,1
7,centralized_global,central_07,False,0.953545,8,0.10,0.85,0.80,2.0,1.0,400,12,1
8,federated_global_cyclic_boosting,federated_07,True,0.960880,8,0.10,0.85,0.55,2.0,1.0,400,12,2
9,federated_global_cyclic_boosting,federated_03,False,0.955990,4,0.10,0.85,0.55,2.0,1.0,400,12,2


### 13.2. Selección del número de clústeres


In [28]:
floor_cluster_tuning = pd.read_csv(FLOOR_CLUSTER_TUNING_PATH)
floor_cluster_tuning = floor_cluster_tuning.sort_values(
    ["modality", "floor_accuracy", "n_clusters"],
    ascending=[True, False, True],
).reset_index(drop=True)

display(
    floor_cluster_tuning[
        [
            "modality",
            "n_clusters",
            "floor_accuracy",
            "coverage",
            "gate_accuracy_diagnostic",
            "selected_candidate_id",
        ]
    ]
)


,modality,n_clusters,floor_accuracy,coverage,gate_accuracy_diagnostic,selected_candidate_id
0,centralized_by_predicted_cluster,3,0.963325,1.0,0.933985,central_00
1,centralized_by_predicted_cluster,4,0.958435,1.0,0.931540,central_00
2,centralized_by_predicted_cluster,6,0.958435,1.0,0.897311,central_00
3,centralized_by_predicted_cluster,2,0.951100,1.0,0.973105,central_00
4,centralized_by_predicted_cluster,5,0.948655,1.0,0.897311,central_00
5,centralized_by_predicted_cluster,7,0.948655,1.0,0.882641,central_00
6,centralized_by_predicted_cluster,8,0.943765,1.0,0.845966,central_00
7,federated_by_predicted_cluster_cyclic_boosting,2,0.963325,1.0,0.973105,federated_07
8,federated_by_predicted_cluster_cyclic_boosting,6,0.955990,1.0,0.897311,federated_07
9,federated_by_predicted_cluster_cyclic_boosting,3,0.941320,1.0,0.933985,federated_07


### 13.3. Resultados de validación y test


In [29]:
print("=== Planta: validación ===")
display(compact_floor_results(floor_results, split="val"))

print("=== Planta: test final ===")
display(compact_floor_results(floor_results, split="test"))


=== Planta: validación ===


,model,modality,n_clusters,floor_accuracy,coverage,gate_accuracy_diagnostic
0,XGBoost,centralized_global,NaN,0.968215,NaN,NaN
1,XGBoost,centralized_by_predicted_cluster,3.0,0.963325,1.0,0.933985
2,XGBoost,federated_by_predicted_cluster_cyclic_boosting,2.0,0.963325,1.0,0.973105
3,XGBoost,federated_global_cyclic_boosting,NaN,0.960880,NaN,NaN


=== Planta: test final ===


,model,modality,n_clusters,floor_accuracy,coverage,gate_accuracy_diagnostic
0,XGBoost,centralized_global,NaN,0.9575,NaN,NaN
1,XGBoost,federated_global_cyclic_boosting,NaN,0.9475,NaN,NaN
2,XGBoost,centralized_by_predicted_cluster,3.0,0.9450,1.0,0.9425
3,XGBoost,federated_by_predicted_cluster_cyclic_boosting,2.0,0.9425,1.0,0.9650


### 13.4. Hiperparámetros seleccionados


In [30]:
with FLOOR_SELECTED_PATH.open("r", encoding="utf-8") as handle:
    selected_floor_hyperparameters = json.load(handle)

print(json.dumps(selected_floor_hyperparameters, indent=2, ensure_ascii=False))


{
  "model": "XGBoost",
  "task": "floor",
  "selection_split": "validation",
  "selection_metric": "floor_accuracy",
  "centralized_global": {
    "candidate_id": "central_00",
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.85,
    "colsample_bytree": 0.55,
    "min_child_weight": 2.0,
    "reg_lambda": 1.0,
    "central_rounds": 400,
    "early_stopping_rounds": 30,
    "federated_cycles": 12,
    "local_boost_rounds": 1,
    "nthread": -1,
    "seed": 42
  },
  "federated_global_cyclic_boosting": {
    "candidate_id": "federated_07",
    "max_depth": 8,
    "learning_rate": 0.1,
    "subsample": 0.85,
    "colsample_bytree": 0.55,
    "min_child_weight": 2.0,
    "reg_lambda": 1.0,
    "central_rounds": 400,
    "early_stopping_rounds": 30,
    "federated_cycles": 12,
    "local_boost_rounds": 2,
    "nthread": -1,
    "seed": 42
  },
  "cluster_protocol": "The centralized and federated winners are reused in their respective cluster-aware strategies; K is selected

## 14. Resumen final


In [31]:
print("=== Resumen final de coordenadas ===")
display(compact_results(coordinate_results, split="test"))

if FLOOR_TASK:
    print("=== Resumen final de planta ===")
    display(compact_floor_results(floor_results, split="test"))


=== Resumen final de coordenadas ===


,model,modality,n_clusters,rmse_2d_m,mean_error_m,median_error_m,p75_error_m,p95_error_m,coverage,gate_accuracy_diagnostic
0,XGBoost,centralized_by_predicted_cluster,6.0,8.357606,5.421491,3.985573,6.669059,12.554969,1.0,0.9075
1,XGBoost,federated_by_predicted_cluster_cyclic_boosting,7.0,8.608230,6.477052,5.112672,8.288602,13.821601,1.0,0.9025
2,XGBoost,centralized_global,NaN,9.490698,6.571708,4.686711,7.563651,19.283758,NaN,NaN
3,XGBoost,federated_global_cyclic_boosting,NaN,14.055876,10.817003,8.284776,13.773378,27.197928,NaN,NaN


=== Resumen final de planta ===


,model,modality,n_clusters,floor_accuracy,coverage,gate_accuracy_diagnostic
0,XGBoost,centralized_global,NaN,0.9575,NaN,NaN
1,XGBoost,federated_global_cyclic_boosting,NaN,0.9475,NaN,NaN
2,XGBoost,centralized_by_predicted_cluster,3.0,0.9450,1.0,0.9425
3,XGBoost,federated_by_predicted_cluster_cyclic_boosting,2.0,0.9425,1.0,0.9650


## 15. Ficheros generados


In [32]:
generated_files = [
    COORD_RESULTS_PATH,
    COORD_HPARAM_TUNING_PATH,
    COORD_CLUSTER_TUNING_PATH,
    COORD_SELECTED_PATH,
]
if FLOOR_TASK:
    generated_files.extend(
        [
            FLOOR_RESULTS_PATH,
            FLOOR_HPARAM_TUNING_PATH,
            FLOOR_CLUSTER_TUNING_PATH,
            FLOOR_SELECTED_PATH,
        ]
    )

display(
    pd.DataFrame(
        [
            {
                "file": str(path.relative_to(ROOT)),
                "exists": path.exists(),
                "size_bytes": path.stat().st_size if path.exists() else 0,
            }
            for path in generated_files
        ]
    )
)


,file,exists,size_bytes
0,results/TUT/tut_top5_xgboost_coordinates_corre...,True,2138
1,results/TUT/tut_top5_xgboost_coordinates_corre...,True,3767
2,results/TUT/tut_top5_xgboost_coordinates_corre...,True,3749
3,results/TUT/tut_top5_xgboost_coordinates_corre...,True,1018
4,results/TUT/tut_top5_xgboost_floor_corrected.csv,True,1564
5,results/TUT/tut_top5_xgboost_floor_corrected_h...,True,2537
6,results/TUT/tut_top5_xgboost_floor_corrected_c...,True,2857
7,results/TUT/tut_top5_xgboost_floor_corrected_s...,True,1018
